### Build Results Fact

In [0]:
dbutils.widgets.text("p_batch_id", "")
v_batch_id = dbutils.widgets.get("p_batch_id")

In [0]:
%run ../00-common/01.environment-config

In [0]:
%run ../00-common/04.gold-helpers

In [0]:
target_table = f"{catalog_name}.{gold_schema}.fact_session_results"

In [0]:
from pyspark.sql import functions as F

#### Step 1 - Read Source Tables
- silver results table
- silver sprints table

In [0]:
results_df = (
    spark
        .table(f"{catalog_name}.{silver_schema}.results")
        .filter((F.col("batch_id") == v_batch_id))
        .withColumn("session_type", F.lit("RACE"))
        .drop("ingestion_timestamp", "race_name", "race_date", "source", "batch_id", "created_timestamp", "updated_timestamp")
)
sprints_df = (
    spark
        .table(f"{catalog_name}.{silver_schema}.sprints")
        .withColumn("session_type", F.lit("SPRINT"))
        .drop("ingestion_timestamp", "race_name", "race_date", "source", "batch_id", "created_timestamp", "updated_timestamp")
)

#### Step 2 - Union Results and Sprints

In [0]:
results_sprints_df = results_df.unionByName(sprints_df)

#### Step 3 - Add derived columns
1. is_win -> Indicates that the driver won the race
1. is_podium -> Indicates that the driver scored a podium result(1,2,3)
1. has_points -> Indicates that the driver has scored points

In [0]:
fact_session_results_df = (
    results_sprints_df
        .withColumn("is_win", F.col("final_position") == 1)
        .withColumn("is_podium", F.col("final_position").between(1, 3))
        .withColumn("has_points", F.col("points") > 0)
)

#### Step 4 - Write the transformed data into `gold` `fact_session_results` table

In [0]:
write_to_gold(
    input_df=fact_session_results_df,
    target_table=target_table,
    merge_condition="t.season = s.season AND t.round = s.round AND t.session_type = s.session_type AND t.constructor_id = s.constructor_id AND t.driver_id = s.driver_id",
    columns_to_update=[
        "grid_position",
        "completed_laps",
        "car_number",
        "points",
        "final_position",
        "final_position_text",
        "status",
        "is_win",
        "is_podium",
        "has_points"
    ]
)